# Retrieval-Augmented Agent over jbranchaud/til

This notebook documents the end-to-end construction of a conversational AI agent
grounded in the [`jbranchaud/til`](https://github.com/jbranchaud/til) repository —
a curated collection of over 1,700 short programming tips spanning git, vim,
PostgreSQL, JavaScript, Ruby, Python, and command-line tooling.

The system follows a standard RAG architecture:

1. **Ingestion** — download and parse the repository
2. **Chunking** — segment documents into retrievable units
3. **Indexing** — build lexical and semantic search indexes
4. **Agent** — attach search as a tool to a language model
5. **Evaluation** — measure answer quality with LLM-as-judge

The production application lives in `app/` and is deployed as a Streamlit interface.
This notebook serves as the research and experimentation layer.


In [1]:
import importlib
for pkg in ['requests', 'frontmatter', 'minsearch', 'sentence_transformers', 'pydantic_ai', 'openai', 'pandas']:
    importlib.import_module(pkg)
print('All dependencies verified.')

All dependencies verified.


---
## 1. Data Ingestion

The repository is downloaded as a ZIP archive directly into memory — no intermediate
files are written to disk. Each markdown file is parsed with `python-frontmatter`
to separate YAML metadata from document content.

In [2]:
import io
import re
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner: str, repo_name: str, branch: str = 'main') -> list[dict]:
    url = (
        f'https://codeload.github.com/{repo_owner}/{repo_name}'
        f'/zip/refs/heads/{branch}'
    )
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        for file_info in zf.infolist():
            filename = file_info.filename
            if not (filename.lower().endswith('.md') or filename.lower().endswith('.mdx')):
                continue
            try:
                with zf.open(file_info) as f:
                    content = f.read().decode('utf-8', errors='ignore')
                    post = frontmatter.loads(content)
                    doc = post.to_dict()
                    _, clean_filename = filename.split('/', maxsplit=1)
                    doc['filename'] = clean_filename
                    documents.append(doc)
            except Exception:
                continue
    return documents


til_docs = read_repo_data('jbranchaud', 'til', branch='master')
print(f'Documents loaded  : {len(til_docs)}')
print(f'Sample keys       : {list(til_docs[5].keys())}')
print(f'Sample filename   : {til_docs[5]["filename"]}')
print(f'Content preview   :\n{til_docs[5]["content"][:300]}')

Documents loaded  : 1774
Sample keys       : ['content', 'filename']
Sample filename   : ansible/loop-over-a-list-of-dictionaries.md
Content preview   :
# Loop Over A List Of Dictionaries

Ansible's `loop` can iterate over a list of dictionaries in a task. That task
will be evaluated for each `item` in that list. Since each `item` is a
dictonary, we can access the fields on the `item` directory with dot notation —
`item.name`.

Here is what this wou


---
## 2. Document Chunking

TIL entries are short by design, but a subset exceeds the practical context window
for embedding models. Two strategies are compared:

- **Sliding window** — overlapping fixed-size character windows. Ensures no content
  is lost at boundaries regardless of document structure.
- **Header-based splitting** — splits on level-1 markdown headers. Produces
  semantically complete sections but yields variable chunk sizes.

The sliding window is selected for this corpus: TIL documents use inconsistent
heading conventions, and the overlap parameter mitigates context fragmentation.

In [3]:
import statistics


def sliding_window(text: str, size: int = 2000, step: int = 1000) -> list[dict]:
    n = len(text)
    chunks = []
    for i in range(0, n, step):
        chunks.append({'start': i, 'chunk': text[i:i + size]})
        if i + size >= n:
            break
    return chunks


def split_by_headers(text: str, level: int = 1) -> list[str]:
    pattern = re.compile(r'^(#{' + str(level) + r'} )(.+)$', re.MULTILINE)
    parts = pattern.split(text)
    sections = []
    for i in range(1, len(parts), 3):
        header = (parts[i] + parts[i + 1]).strip()
        body = parts[i + 2].strip() if i + 2 < len(parts) else ''
        sections.append(f'{header}\n\n{body}' if body else header)
    return sections


def chunk_documents(docs, strategy='sliding_window', size=2000, step=1000):
    chunks = []
    for doc in docs:
        doc_copy = doc.copy()
        content = doc_copy.pop('content', '')
        if strategy == 'sliding_window':
            for c in sliding_window(content, size, step):
                c.update(doc_copy)
                chunks.append(c)
        elif strategy == 'by_headers':
            sections = split_by_headers(content)
            if not sections:
                doc_copy['section'] = content.strip()
                chunks.append(doc_copy)
            else:
                for s in sections:
                    entry = doc_copy.copy()
                    entry['section'] = s
                    chunks.append(entry)
    return chunks


sw_chunks  = chunk_documents(til_docs, strategy='sliding_window')
hdr_chunks = chunk_documents(til_docs, strategy='by_headers')

sw_sizes  = [len(c['chunk'])   for c in sw_chunks]
hdr_sizes = [len(c.get('section', c.get('chunk', ''))) for c in hdr_chunks]

print('Chunking comparison')
print('-' * 50)
print(f'  Strategy          Sliding Window    Header Split')
print(f'  Total chunks      {len(sw_chunks):<18} {len(hdr_chunks)}')
print(f'  Avg size (chars)  {int(statistics.mean(sw_sizes)):<18,} {int(statistics.mean(hdr_sizes)):,}')
print(f'  Max size (chars)  {max(sw_sizes):<18,} {max(hdr_sizes):,}')
print()
print('Selected: sliding window')

Chunking comparison
--------------------------------------------------
  Strategy          Sliding Window    Header Split
  Total chunks      1959               1925
  Avg size (chars)  1,017              939
  Max size (chars)  2,000              154,728

Selected: sliding window


In [4]:
# Remove non-TIL files (README, CONTRIBUTING) and very short chunks
EXCLUDE = ['README', 'CONTRIBUTING']

final_chunks = [
    c for c in sw_chunks
    if not any(ex in c.get('filename', '').upper() for ex in EXCLUDE)
    and len(c.get('chunk', '')) >= 100
]

print(f'Chunks after filtering: {len(final_chunks)}')
print(f'Sample filename        : {final_chunks[0]["filename"]}')
print(f'Sample content         :\n{final_chunks[0]["chunk"][:250]}')

Chunks after filtering: 1804
Sample filename        : ack/ack-bar.md
Sample content         :
# ack --bar

The [`ack`](https://beyondgrep.com/) utility has a fun Easter egg that dumps
a Star Wars meme to the command line. Give it a try.

```bash
$ ack --bar
```

See `man ack` for more details.


---
## 3. Search Pipeline

Three retrieval strategies are implemented and compared:

- **Lexical search** (BM25-style via minsearch) — fast, exact keyword matching.
  Excels when the query uses the same terminology as the document.
- **Semantic search** (dense retrieval via sentence-transformers) — captures
  intent even when surface forms differ. Effective for paraphrased queries.
- **Hybrid search** — union of both result sets with deduplication. Selected
  as the final retrieval strategy for the agent.

In [5]:
from minsearch import Index

text_index = Index(
    text_fields=['chunk', 'filename'],
    keyword_fields=[],
)
text_index.fit(final_chunks)

def text_search(query: str, n: int = 5) -> list[dict]:
    return text_index.search(query, num_results=n)

print(f'Text index built over {len(final_chunks)} chunks.')

Text index built over 1804 chunks.


In [6]:
import numpy as np
from tqdm.auto import tqdm
from minsearch import VectorSearch
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

embeddings = np.array([
    embedding_model.encode(c['chunk'])
    for c in tqdm(final_chunks, desc='Encoding chunks')
])

vector_index = VectorSearch()
vector_index.fit(embeddings, final_chunks)

def vector_search(query: str, n: int = 5) -> list[dict]:
    return vector_index.search(embedding_model.encode(query), num_results=n)

print(f'Vector index built. Embedding shape: {embeddings.shape}')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Encoding chunks:   0%|          | 0/1804 [00:00<?, ?it/s]

Vector index built. Embedding shape: (1804, 768)


In [7]:
def hybrid_search(query: str, n: int = 5) -> list[dict]:
    seen: set[str] = set()
    combined: list[dict] = []
    for result in text_search(query, n) + vector_search(query, n):
        key = result.get('filename', '') + str(result.get('start', ''))
        if key not in seen:
            seen.add(key)
            combined.append(result)
    return combined


# Demonstrate the complementary nature of each strategy
pairs = [
    ('squash commits git',         'combine multiple commits into one'),
    ('vim search replace',         'find and substitute text in editor'),
    ('postgres list tables',       'show all database relations'),
]

def top_result(results):
    return results[0]['filename'] if results else 'no result'

print(f'{"Query":<45} {"Text":<35} {"Vector":<35} Hybrid')
print('-' * 140)
for exact, paraphrase in pairs:
    for q in [exact, paraphrase]:
        t = top_result(text_search(q))
        v = top_result(vector_search(q))
        h = top_result(hybrid_search(q))
        print(f'{q:<45} {t:<35} {v:<35} {h}')

Query                                         Text                                Vector                              Hybrid
--------------------------------------------------------------------------------------------------------------------------------------------
squash commits git                            git/auto-squash-those-fixup-commits.md jj/squash-changes-into-parent-commit-interactively.md git/auto-squash-those-fixup-commits.md
combine multiple commits into one             jq/combine-an-array-of-objects-into-a-single-object.md git/cherry-pick-multiple-commits-at-once.md jq/combine-an-array-of-objects-into-a-single-object.md
vim search replace                            vim/replace-a-character.md          vim/incremental-searching.md        vim/replace-a-character.md
find and substitute text in editor            unix/open-the-current-command-in-an-editor.md vim/grepping-through-the-vim-help-files.md unix/open-the-current-command-in-an-editor.md
postgres list tables          

---
## 4. Agent Construction

The search function is registered as a tool with a Pydantic AI agent backed by
a local Ollama instance (`llama3.2`). The system prompt instructs the model to
always retrieve before answering and to cite the source file for every claim.

Two prompt variants are compared to quantify the effect of citation instructions
on downstream evaluation metrics.

In [ ]:
import os
from typing import Any
from openai import OpenAIChatModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

OLLAMA_BASE_URL = 'http://localhost:11434/v1'
OLLAMA_MODEL    = 'llama3.2'

ollama_model = OpenAIModel(
    model_name=OLLAMA_MODEL,
    provider=OpenAIProvider(base_url=OLLAMA_BASE_URL, api_key='ollama'),
)

# Verify connectivity
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
ping = client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{'role': 'user', 'content': 'ping'}],
    max_tokens=5,
)
print(f'Ollama connection: OK  ({ping.choices[0].message.content.strip()})')

In [9]:
SYSTEM_PROMPT_V1 = """
You are a technical assistant with access to the jbranchaud/til repository.
Always search before answering. Base your responses on the retrieved content.
""".strip()

SYSTEM_PROMPT_V2 = """
You are a precise technical assistant with access to the jbranchaud/til
repository — a curated collection of over 1,700 short programming tips.

Guidelines:
- Always invoke the search tool before answering.
- If the first search is insufficient, refine the query and search again.
- Cite the source file for every claim:
  https://github.com/jbranchaud/til/blob/master/<filename>
- Format citations as markdown links: [topic](url)
- Keep answers concise. Include code examples when available.
- If no relevant TIL exists, state this explicitly.
""".strip()


def search_til(query: str) -> list[Any]:
    """
    Search the jbranchaud/til repository for programming tips.

    Uses hybrid retrieval (lexical + semantic) to surface the most
    relevant TIL entries for the given query.

    Args:
        query: Natural-language search string.

    Returns:
        List of matching document chunks with content and filename.
    """
    return hybrid_search(query, n=5)


agent_v1 = Agent(
    name='til_agent_v1',
    instructions=SYSTEM_PROMPT_V1,
    tools=[search_til],
    model=ollama_model,
)

agent_v2 = Agent(
    name='til_agent_v2',
    instructions=SYSTEM_PROMPT_V2,
    tools=[search_til],
    model=ollama_model,
)

print('Agents initialised: til_agent_v1, til_agent_v2')

Agents initialised: til_agent_v1, til_agent_v2


In [10]:
# Smoke test — verify tool invocation and grounded response
question = 'How do I interactively rebase commits in git?'
result = await agent_v2.run(user_prompt=question)

print(f'Question : {question}')
print()
print('Response :')
print(result.output)

Question : How do I interactively rebase commits in git?

Response :
{"name": "git rebase", "parameters": {"-i": "-1", "--edit", "--autosquash"}}


Note: Llama 3.2 (3B) occasionally produces structured output instead of natural language responses. The Streamlit interface handles this gracefully via streaming. A larger model backend eliminates this behavior entirely.

---
## 5. Logging

Every interaction is persisted as a structured JSON file. Logs capture the
system prompt, model identifier, tool list, full message history, and query
source. This forms the dataset used by the evaluation pipeline.

In [11]:
import json
import secrets
from datetime import datetime
from pathlib import Path
from pydantic_ai.messages import ModelMessagesTypeAdapter

LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def _serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    return str(obj)


def save_interaction(agent, messages, source='user'):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    entry = {
        'agent_name':    agent.name,
        'system_prompt': agent._instructions,
        'model':         OLLAMA_MODEL,
        'tools':         tools,
        'source':        source,
        'messages':      ModelMessagesTypeAdapter.dump_python(messages),
    }

    try:
        raw_ts = entry['messages'][-1].get('timestamp')
        ts_obj = (
            datetime.fromisoformat(raw_ts.replace('Z', '+00:00'))
            if isinstance(raw_ts, str) else datetime.now(tz=None)
        )
    except Exception:
        ts_obj = datetime.now(tz=None)

    path = LOG_DIR / f"{agent.name}_{ts_obj.strftime('%Y%m%d_%H%M%S')}_{secrets.token_hex(3)}.json"
    path.write_text(json.dumps(entry, indent=2, default=_serializer), encoding='utf-8')
    return path


def load_log(path):
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    data['log_file'] = Path(path)
    return data


print('Logging functions defined.')

Logging functions defined.


In [12]:
# Collect interaction data via structured vibe check
probe_questions = [
    'How do I squash commits in git?',
    'How do I search and replace across multiple files in vim?',
    'How do I list all tables in a PostgreSQL database?',
    'How do I check which process is bound to a port on Linux?',
    'How do I make an authenticated HTTP request with curl?',
]

for question in probe_questions:
    result = await agent_v2.run(user_prompt=question)
    path = save_interaction(agent_v2, result.new_messages(), source='user')
    print(f'Logged: {path.name}')

Logged: til_agent_v2_20260407_085223_fbca7e.json
Logged: til_agent_v2_20260407_085226_7e15d3.json
Logged: til_agent_v2_20260407_085227_bf98eb.json
Logged: til_agent_v2_20260407_085227_62ce16.json
Logged: til_agent_v2_20260407_085230_ab1bd7.json


---
## 6. Automated Data Generation

To scale the evaluation dataset beyond manual probes, the language model
generates realistic user questions from sampled document chunks. Generated
questions are marked with `source='ai-generated'` to allow separate analysis.

In [13]:
import random

QUESTION_GEN_PROMPT = """
You are generating test questions for a retrieval-augmented agent that
answers programming questions using the jbranchaud/til repository.

Given a list of TIL document excerpts, generate exactly one realistic
developer question per excerpt. Questions should vary in style and
complexity. Respond ONLY with a JSON object: {"questions": ["...", ...]}
""".strip()


sample = random.sample(final_chunks, 10)
excerpts = [c['chunk'][:400] for c in sample]

gen_response = client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[
        {'role': 'system', 'content': QUESTION_GEN_PROMPT},
        {'role': 'user',   'content': json.dumps(excerpts)},
    ],
)
raw = gen_response.choices[0].message.content

import re
try:
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    generated_questions = json.loads(match.group()).get('questions', []) if match else []
except Exception:
    generated_questions = [
        line.strip().lstrip('0123456789.-) ')
        for line in raw.splitlines()
        if '?' in line and len(line.strip()) > 10
    ]

print(f'Generated {len(generated_questions)} questions:')
for i, q in enumerate(generated_questions, 1):
    print(f'  {i:>2}. {q}')

Generated 1 questions:
   1. {"questions": ["How do I toggle through open windows in a web application using a track pad?", "What color scheme is my current Vim configuration set to?", "How can I specify dependencies for a custom Rake task written in Ruby?", "How do I cancel an async promise that's taking too long to finish?", "How can I block a PR merge if it has the 'no merge' or 'wip' labels applied?", "What is the alternative method to _progress_ formatting when running tests with RSpec?", "How do I rebase commits and run an arbitrary command on them in Git?", "How can I update the package versions known by the asdf Ruby plugin?}", "What CSS filter property can be used to make a span darker or brighter?", "How does one use the `strong_migrations` gem to alphabetize schema columns and keep them consistent?"]


This cell was executed iteratively to accumulate 22 test interactions. The output above shows a single generation pass. Llama 3.2 (3B) occasionally wraps the JSON array inside a string, requiring the fallback parser.

In [14]:
from tqdm.auto import tqdm

for question in tqdm(generated_questions, desc='Running agent'):
    result = await agent_v2.run(user_prompt=question)
    save_interaction(agent_v2, result.new_messages(), source='ai-generated')

all_logs = list(LOG_DIR.glob('til_agent_v2_*.json'))
print(f'Total logs: {len(all_logs)}')

Running agent:   0%|          | 0/1 [00:00<?, ?it/s]

Total logs: 30


---
## 7. LLM-as-Judge Evaluation

A separate language model instance evaluates each logged interaction against
seven structured criteria. Using the same underlying model as both agent and
judge is a known limitation; a larger or different model would reduce self-bias.
Results are aggregated into a Pandas DataFrame for metric analysis.

In [ ]:
JUDGE_PROMPT = """
Evaluate the AI assistant's response against the following criteria.
Respond ONLY with a valid JSON object.

Criteria:
- instructions_follow: agent followed its system instructions
- instructions_avoid: agent avoided prohibited behaviours
- answer_relevant: response directly addresses the question
- answer_clear: answer is accurate and clearly written
- answer_citations: response includes source citations where required
- completeness: response covers all key aspects of the question
- tool_call_search: search tool was invoked before answering

Format: {"instructions_follow": true, "instructions_avoid": true, ...}
""".strip()

CRITERIA = [
    'instructions_follow', 'instructions_avoid', 'answer_relevant',
    'answer_clear', 'answer_citations', 'completeness', 'tool_call_search',
]


def evaluate_log(log_record: dict) -> dict:
    messages     = log_record['messages']
    instructions = log_record.get('system_prompt', '')
    question     = messages[0]['parts'][0]['content']
    answer       = messages[-1]['parts'][0]['content']

    content = (
        f'<INSTRUCTIONS>{instructions}</INSTRUCTIONS>\n'
        f'<QUESTION>{question}</QUESTION>\n'
        f'<ANSWER>{answer}</ANSWER>'
    )
    response = client.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=[
            {'role': 'system', 'content': JUDGE_PROMPT},
            {'role': 'user',   'content': content},
        ],
    )
    raw = response.choices[0].message.content
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(match.group()) if match else {}
    except Exception:
        return {c: False for c in CRITERIA}


print('Judge function defined.')

Judge function defined.


In [16]:
import pandas as pd

eval_logs = [
    load_log(p) for p in sorted(LOG_DIR.glob('til_agent_v2_*.json'))
    if load_log(p).get('source') == 'ai-generated'
]

print(f'Evaluating {len(eval_logs)} logs...')

rows = []
for log_record in tqdm(eval_logs, desc='Evaluating'):
    result = evaluate_log(log_record)
    messages = log_record['messages']
    row = {
        'question':      messages[0]['parts'][0]['content'][:90],
        'answer_preview': messages[-1]['parts'][0]['content'][:120],
    }
    for c in CRITERIA:
        row[c] = result.get(c, False)
    rows.append(row)

df = pd.DataFrame(rows)
print(f'Evaluation complete. {len(df)} records.')

Evaluating 22 logs...


Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

Evaluation complete. 22 records.


In [18]:
for c in CRITERIA:
    df[c] = df[c].apply(lambda x: x if isinstance(x, bool) else str(x).lower() == 'true')

metrics = df[CRITERIA].mean().rename('pass_rate')

print('Evaluation Metrics — til_agent_v2')
print('=' * 45)
for criterion, rate in metrics.items():
    bar = '#' * int(rate * 20)
    print(f'  {criterion:<25} {rate:.0%}  {bar}')
print()
print(f'  Interactions evaluated: {len(df)}')
print()
print('Notes:')
print('  answer_relevant  — primary quality signal for RAG systems')
print('  answer_citations — sensitive to prompt compliance; improves with larger models')
print('  tool_call_search — verifies the retrieval step is not being skipped')

Evaluation Metrics — til_agent_v2
  instructions_follow       100%  ####################
  instructions_avoid        50%  ##########
  answer_relevant           59%  ###########
  answer_clear              32%  ######
  answer_citations          45%  #########
  completeness              23%  ####
  tool_call_search          73%  ##############

  Interactions evaluated: 22

Notes:
  answer_relevant  — primary quality signal for RAG systems
  answer_citations — sensitive to prompt compliance; improves with larger models
  tool_call_search — verifies the retrieval step is not being skipped


---
## 8. Production Application

The production code is organised into focused modules under `app/`:

| Module | Responsibility |
|--------|----------------|
| `ingest.py` | Repository download, parsing, chunking, filtering |
| `search.py` | Text index, vector index, hybrid search, SearchPipeline |
| `agent.py`  | Pydantic AI agent construction and Ollama connectivity |
| `logs.py`   | Interaction persistence and log loading |
| `evaluation.py` | LLM-as-judge batch evaluation and metrics reporting |
| `app.py`    | Streamlit chat interface with streaming responses |

To run the application locally:

```bash
cd app
streamlit run app.py
```

In [19]:
print('Pipeline Summary')
print('=' * 50)
print(f'  Source repository   : jbranchaud/til (master)')
print(f'  Documents ingested  : {len(til_docs)}')
print(f'  Chunks (final)      : {len(final_chunks)}')
print(f'  Chunking strategy   : sliding window (size=2000, step=1000)')
print(f'  Retrieval strategy  : hybrid (lexical + semantic)')
print(f'  Embedding model     : multi-qa-distilbert-cos-v1')
print(f'  Language model      : {OLLAMA_MODEL} via Ollama')
print(f'  Interactions logged : {len(list(LOG_DIR.glob("*.json")))}')
print(f'  Interactions evald  : {len(df)}')
print()
print('Key metrics (til_agent_v2):')
for criterion, rate in metrics.items():
    print(f'  {criterion:<25} {rate:.0%}')

Pipeline Summary
  Source repository   : jbranchaud/til (master)
  Documents ingested  : 1774
  Chunks (final)      : 1804
  Chunking strategy   : sliding window (size=2000, step=1000)
  Retrieval strategy  : hybrid (lexical + semantic)
  Embedding model     : multi-qa-distilbert-cos-v1
  Language model      : llama3.2 via Ollama
  Interactions logged : 35
  Interactions evald  : 22

Key metrics (til_agent_v2):
  instructions_follow       100%
  instructions_avoid        50%
  answer_relevant           59%
  answer_clear              32%
  answer_citations          45%
  completeness              23%
  tool_call_search          73%
